In [8]:
import json
import os
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

# Set base directory and file paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
input_path = os.path.join(BASE_DIR, "output_businesses", "pa_valid_filtered_dining_2.json")
output_path = os.path.join(BASE_DIR, "output_attributes", "attribute_analysis_1.json")

# Load businesses
with open(input_path, 'r', encoding='utf-8') as f:
    businesses = json.load(f)

# Extract attributes
attribute_counts = Counter()  # Counts how many businesses have each attribute
attribute_values = defaultdict(Counter)  # Tracks how often each value appears

for business in businesses:
    attributes = business.get('attributes', {})
    if not isinstance(attributes, dict):  # Handle missing or malformed attributes
        continue

    for attr, value in attributes.items():
        attribute_counts[attr] += 1  # Count occurrences of this attribute
        attribute_values[attr][value] += 1  # Count occurrences of each value

# Sort attributes by frequency
sorted_attributes = sorted(attribute_counts.items(), key=lambda x: x[1], reverse=True)

# Save attribute statistics
attribute_stats = {
    "counts": attribute_counts,
    "values": {k: dict(v) for k, v in attribute_values.items()}
}

output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)  # Ensure directory exists

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(attribute_stats, f, indent=2)

# Display top 20 attributes by frequency
print("\nTop Business Attributes by Frequency:")
for attr, count in sorted_attributes[:5]:
    print(f"{attr}: {count} businesses")

print(f"\nFrom {len(businesses)} businesses extracted {len(sorted_attributes)} attributes saved to: {output_path}")


Top Business Attributes by Frequency:
RestaurantsTakeOut: 6228 businesses
BusinessAcceptsCreditCards: 6117 businesses
BusinessParking: 6114 businesses
RestaurantsDelivery: 5982 businesses
RestaurantsPriceRange2: 5578 businesses

From 6616 businesses extracted 38 attributes saved to: d:\Programming\LLM_RS\output_attributes\attribute_analysis_1.json


Remove low-occurrence attributes

Set the threshold at 5000 Non- None(75% of 6616 businesses) to ensure only widely applicable attributes remain

In [9]:
import json
import os
from collections import defaultdict

# Set file paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
input_path = os.path.join(BASE_DIR, "output_businesses", "pa_valid_filtered_dining_2.json")
filtered_output_path = os.path.join(BASE_DIR, "output_attributes", "attribute_analysis_2.json")

# Set high threshold 
threshold = 5000  

# Initialize attribute counters
attribute_counts = defaultdict(int)  # Total count of occurrences
non_none_counts = defaultdict(int)  # Count of non-"None" values
attribute_values = defaultdict(lambda: defaultdict(int))  # Store unique values per attribute

# Load businesses
with open(input_path, 'r', encoding='utf-8') as f:
    businesses = json.load(f)

# Count attribute occurrences
for business in businesses:
    attributes = business.get("attributes", {})

    if not isinstance(attributes, dict):  # Ensure attributes exist
        continue

    for attr, value in attributes.items():
        attribute_counts[attr] += 1  # Count every appearance of an attribute

        if value != "None":  # Only count non-"None" values
            non_none_counts[attr] += 1

        attribute_values[attr][value] += 1  # Store unique values and their counts

# Filter attributes based on threshold (5,000+ total occurrences)
filtered_attributes = {attr: count for attr, count in non_none_counts.items() if count >= threshold}
filtered_non_none_counts = {attr: non_none_counts[attr] for attr in filtered_attributes}

# Save filtered attributes
output_data = {
    "counts": filtered_attributes,  # Total occurrences
    "non_none_counts": filtered_non_none_counts,  # Non-"None" occurrences
    "values": {k: dict(attribute_values[k]) for k in filtered_attributes}  # Unique values
}

os.makedirs(os.path.dirname(filtered_output_path), exist_ok=True)

with open(filtered_output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2)

# Display filtering results
print(f"Removed {len(attribute_counts) - len(filtered_attributes)} low-occurrence attributes (Threshold: {threshold} businesses)")
print(f"Filtered {len(filtered_attributes)} attributes saved to: {filtered_output_path}")
print(f"Saved 'counts' (total occurrences) and 'non_none_counts' (non-'None' values).")

Removed 27 low-occurrence attributes (Threshold: 5000 businesses)
Filtered 11 attributes saved to: d:\Programming\LLM_RS\output_attributes\attribute_analysis_2.json
Saved 'counts' (total occurrences) and 'non_none_counts' (non-'None' values).


In [12]:
import json
import os
import ast  # To safely convert string dictionaries into real Python dictionaries

# Set file paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
input_path = os.path.join(BASE_DIR, "output_businesses", "pa_valid_filtered_dining_2.json")
filtered_attributes_path = os.path.join(BASE_DIR, "output_attributes", "attribute_analysis_2.json")
output_path = os.path.join(BASE_DIR, "output_businesses", "pa_valid_filtered_dining_3.json")

# Load filtered attributes (above 5,000 threshold)
with open(filtered_attributes_path, "r", encoding="utf-8") as f:
    filtered_data = json.load(f)

filtered_attributes = set(filtered_data["counts"].keys())  # Keep only these attributes

# Attributes to remove manually
UNWANTED_ATTRIBUTES = {"BusinessAcceptsCreditCards", "WiFi", "HasTV"}

# Load businesses
with open(input_path, 'r', encoding='utf-8') as f:
    businesses = json.load(f)

# Process each business entry
for business in businesses:
    attributes = business.get("attributes", {})

    if not isinstance(attributes, dict):  # Ensure attributes is a dictionary
        business["attributes"] = {}
        continue

    # Remove attributes that are NOT in the filtered list
    attributes = {k: v for k, v in attributes.items() if k in filtered_attributes}

    # Remove manually unwanted attributes
    for attr in UNWANTED_ATTRIBUTES:
        attributes.pop(attr, None)  # Remove if exists

    # Fix "BusinessParking"
    parking_str = attributes.get("BusinessParking", "None")
    if isinstance(parking_str, str) and "{" in parking_str:  # Ensure it's a valid dict string
        try:
            parking_dict = ast.literal_eval(parking_str)  # Convert string to dictionary safely
            attributes["BusinessParking"] = "True" if any(parking_dict.values()) else "False"
        except (SyntaxError, ValueError):
            attributes["BusinessParking"] = "Unknown"  # Handle parsing errors

    # Fix "Ambience" (convert to comma-separated string)
    ambience_str = attributes.get("Ambience", "None")
    if isinstance(ambience_str, str) and "{" in ambience_str:  # Ensure it's a valid dict string
        try:
            ambience_dict = ast.literal_eval(ambience_str)  # Convert string to dictionary safely
            cleaned_ambience = sorted([k for k, v in ambience_dict.items() if v is True])  # Extract True values
            attributes["Ambience"] = ", ".join(cleaned_ambience) if cleaned_ambience else "Unknown"
        except (SyntaxError, ValueError):
            attributes["Ambience"] = "Unknown"

    # Remove "None" values from all attributes
    attributes = {k: v for k, v in attributes.items() if v != "None"}

    # Assign the cleaned attributes back to the business
    business["attributes"] = attributes

# Save cleaned dataset
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(businesses, f, indent=2)

print(f"Updated business dataset saved to: {output_path}")
print(f"Removed manually unwanted attributes: {UNWANTED_ATTRIBUTES}")
print(f"Removed 'None' values from attributes.")
print(f"Formatted 'Ambience' as a string.")

Updated business dataset saved to: d:\Programming\LLM_RS\output_businesses\pa_valid_filtered_dining_3.json
Removed manually unwanted attributes: {'WiFi', 'HasTV', 'BusinessAcceptsCreditCards'}
Removed 'None' values from attributes.
Formatted 'Ambience' as a string.


In [13]:
import json
import os
from collections import defaultdict

# Set file paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
input_path = os.path.join(BASE_DIR, "output_businesses", "pa_valid_filtered_dining_3.json")
output_path = os.path.join(BASE_DIR, "output_attributes", "attribute_analysis_3.json")

# Initialize attribute counters
attribute_counts = defaultdict(int)  # Total occurrences of each attribute
non_none_counts = defaultdict(int)  # Non-"None" occurrences
attribute_values = defaultdict(lambda: defaultdict(int))  # Unique values and their counts

# Load businesses
with open(input_path, 'r', encoding='utf-8') as f:
    businesses = json.load(f)

# Compute attribute statistics
for business in businesses:
    attributes = business.get("attributes", {})

    if not isinstance(attributes, dict):  # Ensure attributes exist
        continue

    for attr, value in attributes.items():
        attribute_counts[attr] += 1  # Count every occurrence

        if value != "None":  # Only count non-"None" values
            non_none_counts[attr] += 1

        attribute_values[attr][value] += 1  # Count occurrences of each value

# Save computed attribute statistics
output_data = {
    "counts": attribute_counts,  # Total occurrences
    "non_none_counts": non_none_counts,  # Non-"None" occurrences
    "values": {k: dict(attribute_values[k]) for k in attribute_counts}  # Unique values and their counts
}

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2)

# Display statistics
print(f"Final attribute statistics saved to: {output_path}")
print(f"Total attributes analyzed: {len(attribute_counts)}")
print(f"Total attributes with non-'None' values: {len(non_none_counts)}")

Final attribute statistics saved to: d:\Programming\LLM_RS\output_attributes\attribute_analysis_3.json
Total attributes analyzed: 8
Total attributes with non-'None' values: 8


Many combinations of "ambience"
Merge it into "description"

Manual Check: Wifi, HasTV (to be removed)

Ambience?

Value: None